# ⚡ FlashVSR — Google Colab 影片超解析度放大 (T4 / L4 GPU)

高畫質視訊超解析度放大（Video Super Resolution），原生支援 **Tesla T4 (15GB)** 與 **NVIDIA L4 (22GB)**。

### ✨ 重點特性
- **自適應短邊解析度對齊 (`--resolution`)**：直接指定目標短邊像素（如 720, 1080, 1440），自動計算最適放大倍率與補邊裁切。
- **斷點續傳 (`--resume`)**：支援長影片分段推論與檢查點保存，Colab 斷線隨時可無損接續，完成後 FFmpeg Concat 1 秒無損串接。
- **動態注意力預算 (TE-Speed)**：在 L4 上全速啟用內建免編譯的 `sparse_sage_attention` (INT8 Triton)，T4 自動平滑回退至 SDPA。
- **音訊保留 (`--keep_audio`)**：自動探測並完整保留原始影片所有音軌。
- **統一品質旋鈕 (`--quality 0~10`)**：自動映射至硬體 HEVC NVENC 或 libx264 最高畫質編碼。


In [ ]:
# @title 1. 檢測 GPU 環境與掛載 Google Drive
import torch

if not torch.cuda.is_available():
    raise RuntimeError("❌ 請先在 Colab 選單「執行階段」->「變更執行階段類型」選取 GPU (T4 或 L4)！")

props = torch.cuda.get_device_properties(0)
print(f"✅ 偵測到 GPU: {props.name} ({props.total_memory / (1024**3):.1f} GiB VRAM)")

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# @title 2. 一鍵安裝 FFmpeg 9 與 FlashVSR 相依套件
# 1. 安裝官方 FFmpeg 9 PPA
!add-apt-repository -y ppa:ubuntuhandbook1/ffmpeg9
!apt-get update -qq
!apt-get install -y -qq ffmpeg

# 2. 取得或更新 FlashVSR 專案
!if [ ! -d "/content/FlashVSR" ]; then \
    git clone --depth 1 https://github.com/delvilo/FlashVSR_colab.git /content/FlashVSR; \
else \
    cd /content/FlashVSR && git pull; \
fi

# 3. 安裝 Python 相依套件 (純 PyTorch + TorchCodec，無 torchvision/opencv)
%cd /content/FlashVSR
!pip install -q -r requirements.txt
!pip uninstall -q -y torchvision torchaudio opencv-python opencv-python-headless

# 4. 建立 Google Drive 工作資料夾
!mkdir -p /content/drive/MyDrive/FlashVSR/input
!mkdir -p /content/drive/MyDrive/FlashVSR/models
!mkdir -p /content/drive/MyDrive/FlashVSR/output

print("✅ 環境與 FFmpeg 9 安裝完成！")


In [ ]:
# @title ⚡ (選用) 一鍵安裝預編譯 FlashAttention-2 (僅支援 L4 / A100，T4 不支援)
# 注意：免費 T4 (SM 7.5) 不支援此輪子；L4 (SM 8.9) 預設已採用內建免編譯的 sparse_sage_attention，若需 FA2 可執行本 Cell：
!pip install --no-deps https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.4/flash_attn-2.8.3+cu128torch2.11-cp313-cp313-linux_x86_64.whl


## 🚀 執行影片放大 (依 GPU 硬體選擇專屬最佳作業參數)


In [ ]:
# @title ⚡ 【L4 GPU 專用】極致效能最佳參數 (22GB VRAM · 1080p)
# 特色：
# - 開啟 --memory_policy resident (模型常駐顯存，完全免除 CPU 往返拷貝，提速 2 倍)
# - 全速使用 sparse_sage_attention (硬體原生 SM 8.9 Triton INT8 核心加速 2x)
# - 使用 tiny 模式搭配 TCDecoder，以 1080p 進行超高速放大
# - 自動保留原始影片所有音訊 (--keep_audio)

import os
os.chdir("/content/FlashVSR")

INPUT_VIDEO = "/content/drive/MyDrive/FlashVSR/input/video.mp4"
# 若 Drive 中尚未上傳影片，可取消下一行註解直接使用專案自帶測試範例：
# INPUT_VIDEO = "/home/AI/F1.mp4"

OUTPUT_VIDEO = "/content/drive/MyDrive/FlashVSR/output/l4_flashvsr_1080p.mp4"

!python3 cli_main.py \
  --input "$INPUT_VIDEO" \
  --output "$OUTPUT_VIDEO" \
  --resolution 1080 \
  --mode tiny \
  --memory_policy resident \
  --precision bf16 \
  --attention_mode sparse_sage_attention \
  --keep_audio \
  --quality 9


In [ ]:
# @title 🛡️ 【T4 GPU 專用】穩定零 OOM 最佳參數 (15GB VRAM · 1080p)
# 特色：
# - 自動平滑回退至高速 PyTorch SDPA (避免 Turing SM 7.5 不相容問題)
# - 使用 --memory_policy staged 自動 CPU 分相卸載 (保證 100% 零 OOM)
# - 輕量 tiny-long 流式架構，顯存需求極低 (< 7GB)

import os
os.chdir("/content/FlashVSR")

INPUT_VIDEO = "/content/drive/MyDrive/FlashVSR/input/video.mp4"
OUTPUT_VIDEO = "/content/drive/MyDrive/FlashVSR/output/t4_flashvsr_1080p.mp4"

!python3 cli_main.py \
  --input "$INPUT_VIDEO" \
  --output "$OUTPUT_VIDEO" \
  --resolution 1080 \
  --mode tiny-long \
  --frame_chunk_size 33 \
  --memory_policy staged \
  --precision fp16 \
  --attention_mode auto \
  --keep_audio \
  --quality 8


In [ ]:
# @title 🔄 【長影片斷點續傳】Resumable 1080p 分段接關處理
# 特色：
# - 啟用 --resume 與 --frame_chunk_size 33：每個分塊獨立存為檢查點
# - 若遇到 Colab 逾時或斷線，再次執行同一個 Cell 將自動跳過已完成區塊
# - 全片完成後由 FFmpeg Concat Demuxer 於 1 秒內無損串接並合併音訊

import os
os.chdir("/content/FlashVSR")

INPUT_VIDEO = "/content/drive/MyDrive/FlashVSR/input/long_video.mp4"
OUTPUT_VIDEO = "/content/drive/MyDrive/FlashVSR/output/long_video_resumed_1080p.mp4"

!python3 cli_main.py \
  --input "$INPUT_VIDEO" \
  --output "$OUTPUT_VIDEO" \
  --resolution 1080 \
  --mode tiny-long \
  --frame_chunk_size 33 \
  --resume \
  --memory_policy resident \
  --precision bf16 \
  --attention_mode sparse_sage_attention \
  --keep_audio \
  --quality 8


In [ ]:
# @title 🎨 【極致畫質 (Full Mode)】Wan2.1 VAE 全細節旗艦修復 (L4 專用)
# 特色：
# - 採用 Wan2.1 旗艦 VAE，細節還原度最高
# - 色彩校正 (--color_fix) 確保色彩自然逼真
# - 品質等級設為 10 (CQ=20 / CRF=20)，NVENC 近似視覺無損編碼

import os
os.chdir("/content/FlashVSR")

INPUT_VIDEO = "/content/drive/MyDrive/FlashVSR/input/short_clip.mp4"
OUTPUT_VIDEO = "/content/drive/MyDrive/FlashVSR/output/full_quality_1080p.mp4"

!python3 cli_main.py \
  --input "$INPUT_VIDEO" \
  --output "$OUTPUT_VIDEO" \
  --resolution 1080 \
  --mode full \
  --vae_model Wan2.1 \
  --memory_policy resident \
  --precision bf16 \
  --attention_mode sparse_sage_attention \
  --color_fix \
  --keep_audio \
  --quality 10
